# 06I - Hyperparameter Optimization (Execution-Ready)

Optimize the best-performing model using GridSearchCV. Update the `selected_model` variable after completing 06H.

In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

df=pd.read_csv("american_bankruptcy.csv")
df["target"]=df["status_label"].map({"alive":0,"failed":1})

drop=["status_label","target"]
if "company_name" in df.columns:
    drop.append("company_name")

X=df.drop(columns=drop)
y=df["target"]

num=X.select_dtypes(include="number").columns
cat=X.select_dtypes(exclude="number").columns

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42)

pre=ColumnTransformer([
("num",SimpleImputer(strategy="median"),num),
("cat",Pipeline([
("imp",SimpleImputer(strategy="most_frequent")),
("enc",OneHotEncoder(handle_unknown="ignore"))
]),cat)
])


## Select Model

In [ ]:
# Replace with your best model if different.
selected_model=RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

pipeline=Pipeline([
("preprocessor",pre),
("classifier",selected_model)
])


## Hyperparameter Grid

In [ ]:
param_grid={
'classifier__n_estimators':[100,200],
'classifier__max_depth':[8,12,None],
'classifier__min_samples_leaf':[2,5],
'classifier__min_samples_split':[2,10]
}

grid=GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2
)


## Train Grid Search

In [ ]:
grid.fit(X_train,y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest F1:")
print(grid.best_score_)


## Evaluate Best Model

In [ ]:
best_model=grid.best_estimator_

pred=best_model.predict(X_test)

from sklearn.metrics import classification_report,f1_score,accuracy_score

print(classification_report(y_test,pred))
print("Accuracy:",accuracy_score(y_test,pred))
print("F1:",f1_score(y_test,pred))


## Save Tuned Model

In [ ]:
joblib.dump(best_model,"best_tuned_model.joblib")
print("Saved best_tuned_model.joblib")


## Summary

In [ ]:
print("Hyperparameter tuning completed.")
print("Use this tuned model in Notebook 06J for final production selection.")
